In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%capture
!pip install camel-tools
!pip install -q urlextract
!pip install PyArabic

In [ ]:
import pandas as pd
import re

from camel_tools.utils.normalize import normalize_unicode
from camel_tools.utils.normalize import normalize_alef_maksura_ar
from camel_tools.utils.normalize import normalize_alef_ar
from camel_tools.utils.normalize import normalize_teh_marbuta_ar
from camel_tools.utils.dediac import dediac_ar

from urlextract import URLExtract

from pyarabic.araby import strip_tatweel

In [ ]:
%%capture
!git clone https://github.com/iabufarha/ArSarcasm-v2.git

In [ ]:
arabic_to_latin_numbers = { '٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4', '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9' }

In [ ]:
arabic_to_latin_numbers = { '٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4', '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9' }

extractor = URLExtract()

def clean_text(text):
  text = normalize_unicode(text) # ﷺ -> صلى الله عليه وسلم
  text = normalize_alef_ar(text) # Normalize alef variants to 'ا'
  text = normalize_alef_maksura_ar(text) # Normalize alef maksura 'ى' to yeh 'ي'
  text = normalize_teh_marbuta_ar(text) # Normalize teh marbuta 'ة' to heh 'ه'
  text = dediac_ar(text) # Dediacritization
  text = text.translate(str.maketrans(arabic_to_latin_numbers))

  urls = extractor.find_urls(text)
  for url in urls:
      text = text.replace(url, '')

  text = strip_tatweel(text)    # e.g. الســـــــــــــــــــعودية -> السعودية

  text = re.sub(r'(.)\1{2,}', r'\1\1', text)  # Replace 3+ consecutive chars with just 2

  # text = re.sub(r'[^ء-يa-zA-Z0-9\s\(\)\[\]\{\}]+', ' ', text) # Remove all characters except Arabic letters, English letters, digits, # whitespace, and specific punctuation marks (parentheses, brackets, braces).
  text = re.sub(r'[^ء-ي\s]+', ' ', text)
  text = re.sub(r'\s+', ' ', text).strip() # Collapse multiple spaces into a single space and trim leading/trailing spaces.

  return text


In [ ]:
def print_df_info(name, df):
  print('*'*5)
  acopy = df.copy()
  print(f'len({name}) = {len(df)}')
  print('')
  print(acopy.groupby(['sarcasm']).size().reset_index(name='count'))
  print('')
  print(acopy.groupby(['sentiment']).size().reset_index(name='count'))
  print(acopy.groupby(['dialect']).size().reset_index(name='count'))
  print('')
  print(acopy.groupby('combined_label').size().reset_index(name='count').sort_values(by='count'))
  print('')
  print('*'*5)

In [ ]:
training = pd.read_csv('ArSarcasm-v2/ArSarcasm-v2/training_data.csv')
testing = pd.read_csv('ArSarcasm-v2/ArSarcasm-v2/testing_data.csv')

training.rename(columns={'tweet': 'text'}, inplace=True)
testing.rename(columns={'tweet': 'text'}, inplace=True)

training['text'] = training['text'].apply(clean_text)
testing['text'] = testing['text'].apply(clean_text)

training = training[training['text'].str.strip() != ''].dropna(subset=['text'])
testing = testing[testing['text'].str.strip() != ''].dropna(subset=['text'])

training['combined_label'] = training['sarcasm'].astype(str) + ',' + training['sentiment'].astype(str) + ',' + training['dialect'].astype(str)
testing['combined_label'] = testing['sarcasm'].astype(str) + ',' + testing['sentiment'].astype(str) + ',' + testing['dialect'].astype(str)

print(f'len(training), len(testing), len(test_df) = {len(training), len(testing)}')

len(training), len(testing), len(test_df) = (12547, 3000)


In [ ]:
# Create a validation set that mimics the test distribution
val = pd.DataFrame(columns=training.columns)

# Get the distribution of combined labels in the test set
test_dist = testing['combined_label'].value_counts(normalize=True)

# For each label in the test set, sample a proportional amount from the training set
for label, proportion in test_dist.items():
    if label in training['combined_label'].values:
        # Calculate how many samples we need for this label in validation
        # Assuming we want validation set to be about 15-20% of your original training data
        val_size = int(0.18 * len(training) * proportion)

        # Get all samples with this label
        label_samples = training[training['combined_label'] == label]

        # If we have enough samples, take what we need
        if len(label_samples) > val_size:
            val_samples = label_samples.sample(n=val_size, random_state=42)
        else:
            # If not enough samples, take all we have
            val_samples = label_samples

        # Add to validation set
        val = pd.concat([val, val_samples])

# Remove validation samples from training
new_train = training[~training['text'].isin(val['text'].tolist())]


In [ ]:
new_train = new_train.sample(frac=1, random_state=42)
val = val.sample(frac=1, random_state=42)
testing = testing.sample(frac=1, random_state=42)

In [ ]:
print_df_info('new_train', new_train)
print_df_info('val', val)
print_df_info('testing', testing)

*****
len(new_train) = 10153

   sarcasm  count
0    False   8623
1     True   1530

  sentiment  count
0       NEG   3324
1       NEU   5106
2       POS   1723
  dialect  count
0   egypt   2423
1    gulf    390
2  levant    582
3  magreb     41
4     msa   6717

      combined_label  count
23   True,NEU,magreb      1
13  False,POS,magreb      6
26     True,POS,gulf      7
27   True,POS,levant      7
21     True,NEU,gulf      7
22   True,NEU,levant      8
8   False,NEU,magreb     10
3   False,NEG,magreb     11
18   True,NEG,magreb     13
28      True,POS,msa     15
25    True,POS,egypt     24
24      True,NEU,msa     30
20    True,NEU,egypt     44
1     False,NEG,gulf     53
11    False,POS,gulf     62
16     True,NEG,gulf     96
17   True,NEG,levant    112
12  False,POS,levant    131
2   False,NEG,levant    152
6     False,NEU,gulf    165
7   False,NEU,levant    172
19      True,NEG,msa    426
10   False,POS,egypt    464
0    False,NEG,egypt    486
5    False,NEU,egypt    665
15    Tr

In [ ]:
new_train.to_csv('/content/drive/MyDrive/Project/ArSarcasm-v2/01_data_preparation/train.csv', index=False)
val.to_csv('/content/drive/MyDrive/Project/ArSarcasm-v2/01_data_preparation/val.csv', index=False)
testing.to_csv('/content/drive/MyDrive/Project/ArSarcasm-v2/01_data_preparation/test.csv', index=False)